# Feature Engineering

In [ ]:
import numpy as np
import pandas as pd

df = pd.read_csv('../data/02_interim/03_clean_races_info.csv', parse_dates=['date', 'dob'])

## Features to predict position

#### 1. Adding Driver Age at Race Date to Dataset

In [ ]:
target_idx = df.columns.get_loc('driverId') + 1

df.insert(target_idx, 'driver_age', (df['date'] - df['dob']).dt.days / 365.25)

#### 2. Adding Driver Momentum (Last 3 Races)

In [ ]:
df.insert(target_idx + 1, 'driver_momentum',
          df.groupby('driverId')['positionOrder'].transform(
              lambda x: x.shift(1).rolling(window=3, min_periods=1).mean()
          ))

In [ ]:
df['driver_momentum'] = df['driver_momentum'].fillna(df['grid'])

df[['driverId', 'raceId', 'positionOrder', 'grid', 'driver_momentum']]

#### 3. Adding Constructor/Team Momentum

In [ ]:
# Constructor by race mean
team_avg_by_race = df.groupby(['constructorId', 'date'])['positionOrder'].mean().reset_index()

team_avg_by_race['constructor_momentum'] = team_avg_by_race.groupby('constructorId')['positionOrder'].transform(
    lambda x: x.shift(1).rolling(window=3, min_periods=1).mean()
)

df = pd.merge(df, team_avg_by_race[['constructorId', 'date', 'constructor_momentum']], on=['constructorId', 'date'],
              how='left', validate='many_to_many')

In [ ]:
df['constructor_momentum'] = df['constructor_momentum'].fillna(
    df.groupby(['constructorId', 'raceId'])['grid'].transform('mean')
)

In [ ]:
cols = df.columns.tolist()
col_to_move = cols.pop(cols.index('constructor_momentum'))

target_idx = cols.index('constructorId') + 1

cols.insert(target_idx, col_to_move)

df = df[cols]

#### 4. Adding Driver Skill by Circuit

In [ ]:
df['driver_track_affinity'] = df.groupby(['driverId', 'circuitId'])['positionOrder'].transform(
    lambda x: x.shift(1).expanding(min_periods=1).mean()
)

df['driver_track_affinity'] = df['driver_track_affinity'].fillna(df['driver_momentum'])

#### 5. Adding Constructor Skill by Circuit

In [ ]:
team_circuit_history = df.groupby(['constructorId', 'circuitId', 'date'])['positionOrder'].mean().reset_index()

team_circuit_history['constructor_track_affinity'] = (
    team_circuit_history.groupby(['constructorId', 'circuitId'])['positionOrder'].transform(
        lambda x: x.shift(1).expanding(min_periods=1).mean()
    )
)

df = pd.merge(df, team_circuit_history[['constructorId', 'circuitId', 'date', 'constructor_track_affinity']],
              on=['constructorId', 'circuitId', 'date'], how='left', validate='many_to_many'
              )

df['constructor_track_affinity'] = df['constructor_track_affinity'].fillna(
    df['constructor_momentum'])

## Features to predict DNF

#### Applying is_dnf column

In [ ]:
df['is_dnf'] = np.where(df['status_group'] == 'Finished', 0, 1)

#### 1. Constructor DNF Rate

In [ ]:
team_dnf_history = df.groupby(['constructorId', 'date']).agg(
    dnfs_in_race=('is_dnf', 'sum'),
    cars_in_race=('is_dnf', 'count')
).reset_index()

team_dnf_history['cumulative_dnfs'] = team_dnf_history.groupby('constructorId')['dnfs_in_race'].transform(
    lambda x: x.shift(1).expanding(min_periods=1).sum()
)

team_dnf_history['cumulative_cars'] = team_dnf_history.groupby('constructorId')['cars_in_race'].transform(
    lambda x: x.shift(1).expanding(min_periods=1).sum()
)

team_dnf_history['constructor_dnf_rate'] = team_dnf_history['cumulative_dnfs'] / team_dnf_history['cumulative_cars']

In [ ]:
df = pd.merge(df, team_dnf_history[['constructorId', 'date', 'constructor_dnf_rate']], on=['constructorId', 'date'],
              how='left', validate='many_to_many')

In [ ]:
df['constructor_dnf_rate'] = df['constructor_dnf_rate'].fillna(0)

#### 2. Driver DNF Rate

In [ ]:
df['driver_dnf_rate'] = df.groupby(['driverId'])['is_dnf'].transform(
    lambda x: x.shift(1).expanding(min_periods=1).mean()
)

df['driver_dnf_rate'] = df['driver_dnf_rate'].fillna(0)

## Exporting data

In [ ]:
df.to_csv('../data/02_interim/04_info_after_feature_engineering.csv', index=False)